# Experiment 6 – LIME Stability vs Signal Strength

Goal: measure how much LIME's explanation changes between runs (10 seeds × 500 and 5000 samples) on all 238 Exp 1 + Exp 2 sentences, relate it to signal strength, and test whether LIME's R² or SHAP–LIME disagreement predicts when an explanation is wrong (using Exp 3's counterfactual error). Runtime: Colab → Runtime → Change runtime type → T4 GPU.

Spec: `experiments/lime stability/EXP6_lime_stability.md`.

**Files you need** (Cell 2 asks you to upload them):
- `exp1_sentences.csv` from `experiments/report vs promote/data/`
- `exp3_results.csv` from `experiments/counterfactual swap/results/`

## Cell 1: install, imports and shared helpers (copied from Exp 1, plus Exp 6 additions)

In [ ]:
!pip -q install detoxify shap lime
import importlib.metadata, os, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, shap, lime, sklearn
from matplotlib import colors as mcolors
from detoxify import Detoxify
from lime.lime_text import LimeTextExplainer
from scipy.stats import spearmanr
print("shap", shap.__version__, "| lime", importlib.metadata.version("lime"), "| torch", torch.__version__)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
OUT = "outputs_exp6"; os.makedirs(OUT, exist_ok=True)
BG, FG = "#FFFFFF", "#000000"                     # pink theme (same as Exp 2): white background, black text, pink bars
plt.rcParams.update({"font.size": 14, "axes.titlesize": 16, "axes.labelsize": 14,
                     "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
                     "axes.edgecolor": FG, "axes.labelcolor": FG, "text.color": FG,
                     "xtick.color": FG, "ytick.color": FG, "legend.labelcolor": FG})
COLORS = {"original": "#D6006F", "unbiased": "#FF8FC4", "err": "#333333"}   # side-by-side pair differs strongly in lightness
ATTR_CMAP = mcolors.LinearSegmentedColormap.from_list("grey_white_pink", ["#7A7A7A", "#FFFFFF", "#D6006F"])   # away from toxic = grey, toward = pink

MODELS = {m: Detoxify(m, device=DEVICE) for m in ["original", "unbiased"]}

def proba(texts, model="original", bs=64):
    """P(toxic) for a list of strings, batched so LIME's thousands of samples don't run out of memory."""
    texts = [str(t) for t in texts]
    out = []
    for i in range(0, len(texts), bs):
        out.extend(np.atleast_1d(MODELS[model].predict(texts[i:i+bs])["toxicity"]))
    return np.array(out, dtype=float)

def logit(texts, model="original"):
    """Explain log-odds, not probability, so SHAP contributions are additive (shap-exercises notebook)."""
    p = np.clip(proba(texts, model), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def lime_fn(model="original"):
    """friday-03 pattern: list of strings -> (n, 2) array of [P(non-toxic), P(toxic)]."""
    def f(texts):
        p = proba(texts, model)
        return np.column_stack([1 - p, p])
    return f

def word_tokenizer(s, return_offsets_mapping=True):
    """Word-level tokens that keep their trailing punctuation. shap's built-in regex tokenizer drops the
    sentence-final full stop, so SHAP would explain a different string than the one we score."""
    spans = [m.span() for m in re.finditer(r"\w+\W*|\W+", s)]
    out = {"input_ids": [s[a:b] for a, b in spans]}
    if return_offsets_mapping: out["offset_mapping"] = spans
    return out
MASKER = shap.maskers.Text(word_tokenizer)    # word-level tokens so SHAP and LIME use the same units
def shap_explainer(model="original"):
    return shap.Explainer(lambda t: logit(t, model), MASKER)

def check_efficiency(sv, texts, model, tol=1e-2):
    """Efficiency/additivity axiom (W4-L01 p4, textbook p48): base value + sum of SHAP values = model output."""
    f = logit(texts, model)
    recon = np.array([float(sv[i].base_values) + sv[i].values.sum() for i in range(len(texts))])
    err = np.abs(recon - f).max()
    assert err < tol, f"Efficiency check FAILED, max error {err}"
    print(f"[{model}] efficiency check passed, max error = {err:.2e}")

def shap_word_weights(e):
    """Collapse SHAP tokens into lower-case words; sum repeated words."""
    w = {}
    for tok, v in zip(e.data, e.values):
        k = re.sub(r"\W+", "", tok).lower()
        if k: w[k] = w.get(k, 0.0) + float(v)
    return w

def lime_word_weights(text, model="original", seed=SEED, num_samples=2000):
    """LIME on one sentence. labels=(1,) so weights refer to the TOXIC class (friday-01 sign trap)."""
    expl = LimeTextExplainer(class_names=["non-toxic", "toxic"], random_state=seed)
    n = len(set(re.findall(r"\w+", text)))
    exp = expl.explain_instance(text, lime_fn(model), num_features=n,
                                num_samples=num_samples, labels=(1,))
    return {w.lower(): v for w, v in exp.as_list(label=1)}, exp

def agreement(sw, lw, k=3):
    """Krishna et al. (2022)-style agreement: rank correlation, top-1 match, top-k overlap.
    Ranks are used because SHAP is in log-odds and LIME is in probability, so raw sizes are not comparable."""
    common = sorted(set(sw) & set(lw))
    rho = spearmanr([sw[w] for w in common], [lw[w] for w in common]).correlation if len(common) > 2 else np.nan
    top = lambda d, n: [w for w in sorted(d, key=lambda w: -d[w])[:n]]
    return dict(rho=rho, top1_agree=top(sw, 1) == top(lw, 1),
                top3_overlap=len(set(top(sw, k)) & set(top(lw, k))) / k)

def highlight_png(pairs, title, path):
    """Draw a sentence with each word coloured by its attribution (pink = toward toxic, grey = away).
    Colours are scaled per figure, so compare colour patterns between SHAP and LIME, not exact shades."""
    vmax = max([abs(v) for _, v in pairs] + [1e-9])
    norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax); cmap = ATTR_CMAP
    fig, ax = plt.subplots(figsize=(14, 1.4)); ax.axis("off")
    fig.canvas.draw(); r = fig.canvas.get_renderer(); x = 0.005
    for w, v in pairs:
        t = ax.text(x, 0.5, w, fontsize=20, va="center", transform=ax.transAxes,
                    bbox=dict(boxstyle="round,pad=0.25", fc=cmap(norm(v)), ec="none"))
        x = t.get_window_extent(renderer=r).transformed(ax.transAxes.inverted()).x1 + 0.01
    ax.set_title(title, loc="left", fontsize=16)
    fig.savefig(path, dpi=200, bbox_inches="tight"); plt.close(fig)

# ---- Exp 6 additions ----
import json
from itertools import combinations
from sklearn.metrics import roc_auc_score
SEEDS = list(range(10)); SAMPLE_SIZES = [500, 5000]

# Colour scheme: same pink-and-black theme as the Exp 1 and Exp 2 figures (values from 02_identity_swap.ipynb)
COLORS.update({"control": "#9A8590", "n500": "#A3134F", "n5000": "#FF7AB8",
               "SHAP": "#D6006F", "LIME": "#FF8FC4", "random": "#9A8590"})

def lime_fn(model="original"):
    """Same as Exp 1, but scores LIME's samples in bigger batches (256) so the 4,760 runs finish faster."""
    def f(texts):
        p = proba(texts, model, bs=256)
        return np.column_stack([1 - p, p])
    return f

# Save progress to Google Drive so a Colab disconnect doesn't lose hours of LIME runs.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/xai_exp6"; os.makedirs(CKPT_DIR, exist_ok=True)
else:
    CKPT_DIR = OUT
CKPT = f"{CKPT_DIR}/exp6_lime_runs.csv"     # one row per (sentence, n, seed); reused if the notebook is restarted
print("checkpoint file:", CKPT)

## Cell 2: the sentences (48 from Exp 1 + 190 from Exp 2)

In [ ]:
from google.colab import files
for f in ["exp1_sentences.csv", "exp3_results.csv"]:
    if not os.path.exists(f):
        print(f"Please upload {f}"); files.upload()
assert os.path.exists("exp1_sentences.csv") and os.path.exists("exp3_results.csv")

exp1 = pd.read_csv("exp1_sentences.csv")
TEMPLATES = ["I am a {} person.", "My neighbour is {} and she is lovely.",
             "{} people are welcome here.", "My best friend is {}.",
             "I grew up in a {} family.", "Being {} is completely normal.",
             "We hired a {} engineer last week.", "She is a proud {} woman.",
             "He is {} and works as a teacher.", "Our new {} classmate joined the team."]
TERMS = ["gay","lesbian","transgender","black","white","asian","muslim","jewish",
         "christian","immigrant","deaf","blind","disabled","poor","rural"]
CONTROLS = ["tall","young","local","quiet"]
exp2 = pd.DataFrame([(t.format(w), w, "control" if w in CONTROLS else "identity") for t in TEMPLATES for w in TERMS + CONTROLS],
                    columns=["text","term","group"])
S = pd.concat([pd.DataFrame({"text": exp1.text, "term": np.nan, "group": exp1.condition, "source": "exp1"}),
               exp2.assign(source="exp2")], ignore_index=True)
print(len(S), "sentences (expect 238)")
print(S.groupby(["source","group"]).size())

## Cell 3: SHAP once per sentence (signal strength + the comparison for agreement)

In [ ]:
SV = shap_explainer("original")(list(S.text), max_evals=500, batch_size=32)
check_efficiency(SV, list(S.text), "original")
S["logit"] = logit(S.text, "original")
S["base"] = [float(SV[i].base_values) for i in range(len(S))]
S["signal"] = (S.logit - S.base).abs()                       # how far the score is from 'everything removed'
SHAP_W = [shap_word_weights(SV[i]) for i in range(len(S))]
S["shap_top"] = [max(w, key=w.get) for w in SHAP_W]
S.to_csv(f"{OUT}/exp6_sentences.csv", index=False)
print(S.groupby(["source","group"]).signal.describe().round(2))

## Cell 4: LIME, 10 seeds × 2 sample sizes, with checkpointing

This is the long cell. It prints progress every 10 sentences. If Colab disconnects, reconnect and run all cells again: finished runs are skipped.

In [ ]:
import time
done = set()
if os.path.exists(CKPT):
    old = pd.read_csv(CKPT); done = set(zip(old.i, old.n, old.seed))
print(f"{len(done)} runs already done, {len(S) * 20 - len(done)} to go")
t0 = time.time()
for i, text in enumerate(S.text):
    new = []
    for n in SAMPLE_SIZES:
        for seed in SEEDS:
            if (i, n, seed) in done: continue
            lw, exp = lime_word_weights(text, "original", seed=seed, num_samples=n)
            new.append(dict(i=i, n=n, seed=seed, top=max(lw, key=lw.get), r2=exp.score,
                            weights=json.dumps(lw)))
    if new:
        pd.DataFrame(new).to_csv(CKPT, mode="a", header=not os.path.exists(CKPT), index=False)
    if i % 10 == 0: print(f"{i}/{len(S)} sentences done  ({(time.time() - t0) / 60:.1f} min)")
runs = pd.read_csv(CKPT).drop_duplicates(["i", "n", "seed"])
runs["weights"] = runs.weights.apply(json.loads)
runs.to_csv(f"{OUT}/exp6_lime_runs.csv", index=False)
print(len(runs), "LIME runs (expect", len(S) * 20, ")")

## Cell 5: stability per sentence

- `flip_rate`: share of runs whose top word differs from the most common top word.
- `distinct_top`: how many different top words appeared across the 10 runs.
- `rank_stability`: mean Spearman between every pair of the 10 runs.
- `r2`: mean LIME fit (local fidelity).
- `agree`: share of runs whose top word matches SHAP's top word.

In [ ]:
def pairwise_rho(ws):
    vals = []
    for a, b in combinations(ws, 2):
        common = sorted(set(a) & set(b))
        if len(common) > 2: vals.append(spearmanr([a[k] for k in common], [b[k] for k in common]).correlation)
    return np.nanmean(vals) if vals else np.nan
rows = []
for (i, n), g in runs.groupby(["i","n"]):
    mode = g.top.mode()[0]
    rows.append(dict(i=i, n=n, flip_rate=float((g.top != mode).mean()), distinct_top=g.top.nunique(),
                     rank_stability=pairwise_rho(list(g.weights)), r2=g.r2.mean(),
                     agree=float((g.top == S.shap_top[i]).mean())))
ST = pd.DataFrame(rows).merge(S, left_on="i", right_index=True)
ST.to_csv(f"{OUT}/exp6_stability_by_sentence.csv", index=False)
for n in SAMPLE_SIZES:
    d = ST[ST.n == n]
    print(f"n={n}: top word changed in {(d.distinct_top > 1).mean():.0%} of sentences; "
          f"rho(signal, rank_stability) = {spearmanr(d.signal, d.rank_stability, nan_policy='omit').correlation:.2f}; "
          f"rho(signal, r2) = {spearmanr(d.signal, d.r2).correlation:.2f}")

## Cell 6: do R² and disagreement warn us when the explanation is wrong? (uses Exp 3)

For the 150 identity sentences, the **error** = |counterfactual effect − SHAP estimate| from Exp 3. We test whether sentences with low R² or SHAP–LIME disagreement are the ones with big errors.

In [ ]:
e3 = pd.read_csv("exp3_results.csv")[["text","cf_original","est_original"]]
e3["error"] = (e3.cf_original - e3.est_original).abs()
W = ST[(ST.n == 5000) & (ST.group == "identity")].merge(e3, on="text")
print(len(W), "identity sentences matched with Exp 3 (expect 150)")
big = W.error > W.error.quantile(0.75)                     # 'explanation is badly wrong' = top quarter of errors
def safe_auc(y, s):
    return roc_auc_score(y, s) if pd.Series(s).nunique() > 1 else np.nan   # AUC undefined if the score never varies
warn = pd.DataFrame({
    "rho with error": {"low R2 (1-R2)": spearmanr(1 - W.r2, W.error).correlation,
                       "SHAP-LIME disagreement": spearmanr(1 - W.agree, W.error).correlation,
                       "weak signal (-signal)": spearmanr(-W.signal, W.error).correlation},
    "AUC for flagging the worst 25%": {"low R2 (1-R2)": safe_auc(big, 1 - W.r2),
                                       "SHAP-LIME disagreement": safe_auc(big, 1 - W.agree),
                                       "weak signal (-signal)": safe_auc(big, -W.signal)}})
print(warn.round(3)); warn.to_csv(f"{OUT}/exp6_warning_signs.csv")

*AUC 0.5 = no better than guessing; above about 0.7 = a useful warning sign. If neither R² nor disagreement gets there, the presentation's "report R²" / "use disagreement as a flag" recommendations need softening in the paper. If SHAP–LIME disagreement shows NaN, SHAP and LIME agreed on every identity sentence, so disagreement can't flag anything here: say that in the write-up.*

## Cell 7: Figure 2 (paper Fig. 2)

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.8))
for n, c in [(500, COLORS["n500"]), (5000, COLORS["n5000"])]:
    d = ST[ST.n == n]
    ax.scatter(d.signal, 1 - d.rank_stability, s=8, alpha=0.7, color=c, label=f"{n} samples")
ax.set_xscale("symlog"); ax.set_xlabel("Signal strength |logit − base| (log-odds)", fontsize=8)
ax.set_ylabel("LIME instability (1 − ρ across seeds)", fontsize=8)
ax.legend(frameon=False, fontsize=7); ax.tick_params(labelsize=7); ax.spines[["top","right"]].set_visible(False)
fig.tight_layout(); fig.savefig(f"{OUT}/fig_exp6_instability_vs_signal.pdf"); fig.savefig(f"{OUT}/fig_exp6_instability_vs_signal.png", dpi=300)
plt.show()

## Cell 8: results summary (numbers to quote in the paper)

In [ ]:
print("=== EXPERIMENT 6 SUMMARY ===")
summ = ST.groupby(["n","source","group"]).agg(flip_rate=("flip_rate","mean"), changed=("distinct_top", lambda s: (s > 1).mean()),
                                             rank_stability=("rank_stability","mean"), r2=("r2","mean"),
                                             agree=("agree","mean")).round(3)
print(summ); summ.to_csv(f"{OUT}/exp6_summary.csv")
for n in SAMPLE_SIZES:
    d = ST[ST.n == n]
    print(f"n={n}: top word changed in {(d.distinct_top > 1).mean():.0%} of {len(d)} sentences; "
          f"mean R2 {d.r2.mean():.2f}; rho(signal, instability) = "
          f"{spearmanr(d.signal, 1 - d.rank_stability, nan_policy='omit').correlation:.2f}")
ref = ST[ST.text == "Being gay is completely normal."]
print("Slide 8 check, 'Being gay is completely normal.' distinct top words:", ref.set_index("n").distinct_top.to_dict(), "(expect 1 and 1)")
print(warn.round(3))

## Cell 9: download everything

In [ ]:
!zip -qr outputs_exp6.zip outputs_exp6
from google.colab import files; files.download("outputs_exp6.zip")

## After running

1. Unzip the outputs into `experiments/lime stability/results/`.
2. Go through the checks in Part 6 of the spec: 4,760 runs; efficiency passed; the slide 8 sentence has 1 distinct top word; Cell 6 matched 150 sentences.
3. Copy the Cell 8 summary into a journal entry. Note whether R² and disagreement work as warning signs.